# 01 — Exploratory Data Analysis
**E-Commerce Product Recommendation Engine**

This notebook explores the synthetic dataset generated by `data/generate_data.py`.

Sections:
1. Load Data
2. Users EDA
3. Products EDA
4. Interactions EDA
5. Cold-Start Analysis
6. Data Quality
7. Key Findings Summary

In [ ]:
%matplotlib inline

import sys
import os
from pathlib import Path

# ── Make sure src/ is on the path regardless of CWD ──────────────────────────
REPO_ROOT = Path(os.getcwd())
# Walk up until we find src/
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'src').is_dir():
        REPO_ROOT = parent
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.preprocessing import load_raw_data, clean_interactions

# Global style
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

# Figure output directory
FIGURES_DIR = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root : {REPO_ROOT}')
print(f'Figures   : {FIGURES_DIR}')

---
## 1. Load Data

In [ ]:
DATA_DIR = REPO_ROOT / 'data' / 'raw'

users_df, products_df, interactions_df = load_raw_data(str(DATA_DIR))
interactions_df = clean_interactions(interactions_df)

print(f'Users        : {users_df.shape}')
print(f'Products     : {products_df.shape}')
print(f'Interactions : {interactions_df.shape}')

print('\n--- Users (first 3 rows) ---')
display(users_df.head(3))

print('\n--- Products (first 3 rows) ---')
display(products_df.head(3))

print('\n--- Interactions (first 3 rows) ---')
display(interactions_df.head(3))

---
## 2. Users EDA

In [ ]:
# ── Age Distribution ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(users_df['age'], bins=20, kde=True, color='steelblue', ax=ax)
ax.set_title('User Age Distribution', fontsize=14)
ax.set_xlabel('Age')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'users_age_distribution.png', dpi=120)
plt.show()
print(users_df['age'].describe().to_string())

In [ ]:
# ── Gender Split ──────────────────────────────────────────────────────────────
gender_counts = users_df['gender'].value_counts()

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    gender_counts.values,
    labels=gender_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('muted', len(gender_counts)),
)
ax.set_title('Gender Split', fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'users_gender_split.png', dpi=120)
plt.show()
print(gender_counts.to_string())

In [ ]:
# ── Top-10 Locations ──────────────────────────────────────────────────────────
top_locations = users_df['location'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=top_locations.values, y=top_locations.index, palette='Blues_r', ax=ax)
ax.set_title('Top-10 User Locations', fontsize=14)
ax.set_xlabel('Number of Users')
ax.set_ylabel('Location')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'users_top10_locations.png', dpi=120)
plt.show()
print(top_locations.to_string())

In [ ]:
# ── User Segments ─────────────────────────────────────────────────────────────
segment_counts = users_df['user_segment'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=segment_counts.index, y=segment_counts.values, palette='Set2', ax=ax)
ax.set_title('User Segments', fontsize=14)
ax.set_xlabel('Segment')
ax.set_ylabel('Count')
plt.xticks(rotation=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'users_segments.png', dpi=120)
plt.show()
print(segment_counts.to_string())

---
## 3. Products EDA

In [ ]:
# ── Category Distribution ─────────────────────────────────────────────────────
cat_counts = products_df['category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=cat_counts.index, y=cat_counts.values, palette='tab10', ax=ax)
ax.set_title('Product Category Distribution', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Number of Products')
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'products_category_distribution.png', dpi=120)
plt.show()
print(cat_counts.to_string())

In [ ]:
# ── Price by Category (Boxplot) ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
category_order = (
    products_df.groupby('category')['price']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)
sns.boxplot(
    data=products_df,
    x='category',
    y='price',
    order=category_order,
    palette='tab10',
    ax=ax,
)
ax.set_title('Price Distribution by Category', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Price ($)')
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'products_price_by_category.png', dpi=120)
plt.show()
print(products_df.groupby('category')['price'].describe().round(2).to_string())

In [ ]:
# ── Top Brands ────────────────────────────────────────────────────────────────
top_brands = products_df['brand'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='rocket_r', ax=ax)
ax.set_title('Top-15 Brands by Product Count', fontsize=14)
ax.set_xlabel('Number of Products')
ax.set_ylabel('Brand')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'products_top_brands.png', dpi=120)
plt.show()
print(top_brands.to_string())

In [ ]:
# ── Rating Distribution ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(products_df['avg_rating'], bins=20, kde=True, color='coral', ax=ax)
ax.set_title('Average Product Rating Distribution', fontsize=14)
ax.set_xlabel('Average Rating')
ax.set_ylabel('Count')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'products_rating_distribution.png', dpi=120)
plt.show()
print(products_df['avg_rating'].describe().to_string())

---
## 4. Interactions EDA

In [ ]:
# ── Interaction Type Counts ───────────────────────────────────────────────────
itype_counts = interactions_df['interaction_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=itype_counts.index, y=itype_counts.values, palette='Set1', ax=ax)
ax.set_title('Interaction Type Counts', fontsize=14)
ax.set_xlabel('Interaction Type')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'interactions_type_counts.png', dpi=120)
plt.show()
print(itype_counts.to_string())

In [ ]:
# ── Timeline: Interactions per Day ───────────────────────────────────────────
daily = (
    interactions_df
    .set_index('timestamp')
    .resample('D')
    .size()
    .rename('count')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily['timestamp'], daily['count'], linewidth=1.2, color='teal')
ax.fill_between(daily['timestamp'], daily['count'], alpha=0.2, color='teal')
ax.set_title('Interactions per Day', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Interactions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'interactions_timeline.png', dpi=120)
plt.show()
print(f'Date range: {daily["timestamp"].min().date()} → {daily["timestamp"].max().date()}')
print(f'Avg interactions/day: {daily["count"].mean():.1f}')

In [ ]:
# ── Top-20 Most Popular Items ─────────────────────────────────────────────────
top_items = interactions_df['item_id'].value_counts().head(20)

# Merge product names for readability
item_name_map = products_df.set_index('item_id')['name'].to_dict()
top_items_df = (
    top_items
    .reset_index()
    .rename(columns={'index': 'item_id', 'item_id': 'interaction_count', 'count': 'interaction_count'})
)
# Handle different pandas versions
if 'item_id' not in top_items_df.columns:
    top_items_df = top_items.rename_axis('item_id').reset_index(name='interaction_count')
top_items_df['name'] = top_items_df['item_id'].map(item_name_map).fillna(top_items_df['item_id'])
top_items_df['label'] = top_items_df['item_id'] + ' — ' + top_items_df['name'].str[:30]

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=top_items_df,
    x='interaction_count',
    y='label',
    palette='viridis',
    ax=ax,
)
ax.set_title('Top-20 Most Popular Items (by Interaction Count)', fontsize=14)
ax.set_xlabel('Total Interactions')
ax.set_ylabel('Item')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'interactions_top20_items.png', dpi=120)
plt.show()

In [ ]:
# ── Interaction Distribution per User (Histogram) ─────────────────────────────
user_interaction_counts = interactions_df.groupby('user_id').size()

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(user_interaction_counts, bins=40, kde=True, color='mediumpurple', ax=ax)
ax.set_title('Distribution of Interactions per User', fontsize=14)
ax.set_xlabel('Number of Interactions')
ax.set_ylabel('Number of Users')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'interactions_per_user_histogram.png', dpi=120)
plt.show()
print(user_interaction_counts.describe().to_string())

---
## 5. Cold-Start Analysis

In [ ]:
COLD_START_THRESHOLD = 5

# ── Cold-start users ─────────────────────────────────────────────────────────
user_counts = interactions_df.groupby('user_id').size()
cold_users  = user_counts[user_counts < COLD_START_THRESHOLD]
print(f'Users with < {COLD_START_THRESHOLD} interactions (cold-start): {len(cold_users)} '
      f'({100 * len(cold_users) / len(user_counts):.1f}% of active users)')

total_users = len(users_df)
inactive_users = total_users - len(user_counts)
print(f'Users with 0 interactions (never active)   : {inactive_users} '
      f'({100 * inactive_users / total_users:.1f}% of all users)')

# ── Cold-start items ─────────────────────────────────────────────────────────
item_counts = interactions_df.groupby('item_id').size()
cold_items  = item_counts[item_counts < COLD_START_THRESHOLD]
print(f'\nItems with < {COLD_START_THRESHOLD} interactions (cold-start): {len(cold_items)} '
      f'({100 * len(cold_items) / len(item_counts):.1f}% of active items)')

total_items = len(products_df)
inactive_items = total_items - len(item_counts)
print(f'Items with 0 interactions (never seen)      : {inactive_items} '
      f'({100 * inactive_items / total_items:.1f}% of all items)')

# ── Visualise cold-start severity ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# User CDF
user_counts_sorted = user_counts.sort_values()
axes[0].plot(user_counts_sorted.values, np.linspace(0, 1, len(user_counts_sorted)))
axes[0].axvline(COLD_START_THRESHOLD, color='red', linestyle='--', label=f'threshold={COLD_START_THRESHOLD}')
axes[0].set_title('CDF: Interactions per User', fontsize=13)
axes[0].set_xlabel('Interactions')
axes[0].set_ylabel('Cumulative Fraction of Users')
axes[0].legend()

# Item CDF
item_counts_sorted = item_counts.sort_values()
axes[1].plot(item_counts_sorted.values, np.linspace(0, 1, len(item_counts_sorted)), color='coral')
axes[1].axvline(COLD_START_THRESHOLD, color='red', linestyle='--', label=f'threshold={COLD_START_THRESHOLD}')
axes[1].set_title('CDF: Interactions per Item', fontsize=13)
axes[1].set_xlabel('Interactions')
axes[1].set_ylabel('Cumulative Fraction of Items')
axes[1].legend()

fig.suptitle('Cold-Start Analysis', fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'cold_start_analysis.png', dpi=120)
plt.show()

---
## 6. Data Quality

In [ ]:
print('=' * 55)
print('MISSING VALUES')
print('=' * 55)

for name, df in [('users', users_df), ('products', products_df), ('interactions', interactions_df)]:
    mv = df.isnull().sum()
    mv = mv[mv > 0]
    if mv.empty:
        print(f'  [{name}] No missing values ✓')
    else:
        print(f'  [{name}]')
        print(mv.to_string())

print()
print('=' * 55)
print('DUPLICATE ROWS')
print('=' * 55)
for name, df in [('users', users_df), ('products', products_df), ('interactions', interactions_df)]:
    n_dup = df.duplicated().sum()
    print(f'  [{name}] {n_dup} duplicate rows')

print()
print('=' * 55)
print('TIMESTAMP RANGE')
print('=' * 55)
ts = interactions_df['timestamp']
print(f'  Min : {ts.min()}')
print(f'  Max : {ts.max()}')
print(f'  Span: {(ts.max() - ts.min()).days} days')

In [ ]:
# ── Missing-value heatmap ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, df) in zip(axes, [('Users', users_df), ('Products', products_df), ('Interactions', interactions_df)]):
    mv_pct = df.isnull().mean() * 100
    colors = ['salmon' if v > 0 else 'lightgreen' for v in mv_pct.values]
    ax.barh(mv_pct.index, mv_pct.values, color=colors)
    ax.set_title(f'{name}: Missing % per Column', fontsize=11)
    ax.set_xlabel('Missing (%)')
    ax.set_xlim(0, max(mv_pct.max() + 2, 5))

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'data_quality_missing_values.png', dpi=120)
plt.show()

---
## 7. Key Findings Summary

### Dataset Overview
| Dataset | Rows | Key Features |
|---|---|---|
| Users | 500 | age, gender, location, user_segment |
| Products | 200 | 10 categories, price, avg_rating |
| Interactions | 15,000+ | view / cart / purchase, 180-day span |

### Users
- **Age range** spans young adults through seniors; distribution appears roughly uniform or slightly skewed toward 25–45.
- **Gender** split is roughly balanced (Male / Female / Other).
- **Segments** (e.g., budget, mid-tier, premium) allow segment-aware recommendations.
- Top locations are concentrated in a small number of cities — location-based personalisation may be feasible.

### Products
- **10 categories** are roughly equally represented (≈ 20 products each).
- **Price** variance is high within categories (Electronics, Automotive tend higher; Books, Food lower) — price normalisation is essential.
- **Ratings** cluster between 3.0–4.5; left-skewed tail of poorly-rated items.

### Interactions
- **Power-law popularity**: top 20 items account for a disproportionate share of interactions (Zipf skew).
- Interaction funnel: views >> carts > purchases (healthy conversion-funnel shape).
- **Activity is fairly stable** over the 180-day window with weekday peaks.
- Per-user interaction count is right-skewed: most users have moderate activity, a small group are heavy users.

### Cold-Start
- A non-trivial fraction of users have fewer than 5 interactions → **user cold-start** is a real challenge.
- Some items have fewer than 5 interactions → **item cold-start** addressed via content-based fallback using TF-IDF + price features.

### Data Quality
- After `clean_interactions`: no duplicate rows, all timestamps parseable.
- Ratings column has NaN for non-purchase interaction types (expected; fill with 0 or handle per model).

### Modelling Implications
1. **Collaborative filtering** is viable given sufficient density for most users.
2. **Content-based filtering** covers cold-start items via `build_item_features()`.
3. **Hybrid model** (CF + CBF) is recommended.
4. Chronological split with 30-day hold-out avoids data leakage.
5. Weighted interaction scores (view=1, cart=3, purchase=5) better reflect true user preference than raw counts.